In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier 
from sklearn.metrics import accuracy_score , classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
TRAIN_PATH = "../../integrated/data/processed/train(7).csv"
VALIDATION_PATH = "../../integrated/data/processed/validation(7).csv"
TEST_PATH = "../../integrated/data/processed/test(7).csv"

In [3]:
train_df= pd.read_csv(TRAIN_PATH)
validation_df= pd.read_csv(VALIDATION_PATH)
test_df= pd.read_csv(TEST_PATH)

X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]

X_validation = validation_df.drop(columns=["label"])
y_validation = validation_df["label"]

X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]

del train_df, validation_df, test_df

In [ ]:
#drop turn_id and conv_id columns from X_train, X_validation, and X_test 
X_train_ids = X_train[["turn_id", "conv_id"]]
X_validation_ids = X_validation[["turn_id", "conv_id"]]
X_test_ids = X_test[["turn_id", "conv_id"]]

X_train = X_train.drop(columns=["turn_id", "conv_id"])
X_validation = X_validation.drop(columns=["turn_id", "conv_id"])
X_test = X_test.drop(columns=["turn_id", "conv_id"])

In [5]:
print("Training set shape:", X_train.shape)
print("Validation set shape:", X_validation.shape)
print("Test set shape:", X_test.shape)

Training set shape: (8668, 26)
Validation set shape: (2844, 26)
Test set shape: (2872, 26)


In [ ]:
def evaluate_model(y_pred, y_true):
    """evaluation for any model it returns accuracy, classification report and confusion matrix"""
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    return acc, report, cm

In [7]:
# grid search on xgboost

grid_search_params = {
    'n_estimators': [100, 200, 300, 500],    
    'max_depth': [3, 5, 7,10],
    'learning_rate': [0.01, 0.1, 0.2,0.005],
    'random_state': [42],   
    'n_jobs': [-1]
}

xgb = XGBClassifier()
grid_search = GridSearchCV(estimator=xgb, param_grid=grid_search_params, cv=3, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)   

print("Best Hyperparameters:", grid_search.best_params_)


val_acc, val_report, val_cm = evaluate_model(grid_search.predict(X_validation), y_validation)
test_acc, test_report, test_cm = evaluate_model(grid_search.predict(X_test), y_test)
print("Validation Accuracy:", val_acc)
print("Validation Classification Report:\n", val_report)

print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)






Fitting 3 folds for each of 64 candidates, totalling 192 fits
Best Hyperparameters: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100, 'n_jobs': -1, 'random_state': 42}
Validation Accuracy: 0.929324894514768
Validation Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.95      0.96      2449
           1       0.73      0.77      0.75       395

    accuracy                           0.93      2844
   macro avg       0.85      0.86      0.86      2844
weighted avg       0.93      0.93      0.93      2844

Test Accuracy: 0.9209610027855153
Test Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.96      0.95      2464
           1       0.74      0.68      0.71       408

    accuracy                           0.92      2872
   macro avg       0.84      0.82      0.83      2872
weighted avg       0.92      0.92      0.92      2872



In [8]:
# grid search for random forest
grid_search_params = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

model = RandomForestClassifier()
grid_search = GridSearchCV(estimator=model, param_grid=grid_search_params, cv=3, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)   

print("Best Hyperparameters:", grid_search.best_params_)


val_acc, val_report, val_cm = evaluate_model(grid_search.predict(X_validation), y_validation)
test_acc, test_report, test_cm = evaluate_model(grid_search.predict(X_test), y_test)
print("Validation Accuracy:", val_acc)
print("Validation Classification Report:\n", val_report)

print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)





Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best Hyperparameters: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
Validation Accuracy: 0.9191279887482419
Validation Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.96      0.95      2449
           1       0.72      0.69      0.70       395

    accuracy                           0.92      2844
   macro avg       0.83      0.82      0.83      2844
weighted avg       0.92      0.92      0.92      2844

Test Accuracy: 0.9136490250696379
Test Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.96      0.95      2464
           1       0.74      0.61      0.67       408

    accuracy                           0.91      2872
   macro avg       0.84      0.79      0.81      2872
weighted avg       0.91      0.91      0.91      2872

